In [1]:
#!pip install spacy scikit-learn seaborn matplotlib plotly nbformat

In [2]:
#!python -m spacy download en_core_web_sm

# 2. Data preparation

## Text lemmatization with spacy

In [3]:
import spacy
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

nlp = spacy.load('en_core_web_sm')

In [4]:
dataset = pd.read_csv('.data/dataset.csv')

In [5]:
all_song_lyrics = []
for song in tqdm(dataset['Lyrics'].tolist()):
    clean_lyrics = []
    for line in song.split('\n'):
        temp = []
        d = nlp(line)
        for word in d:
            if not word.is_punct:
                temp.append(word.lemma_)
        clean_lyrics.append(' '.join(temp))
    all_song_lyrics.append(clean_lyrics)

100%|██████████| 379/379 [00:52<00:00,  7.25it/s]


In [6]:
dataset['Lyrics_lemmatized'] = all_song_lyrics
dataset.Lyrics_lemmatized.iloc[0][:10]

['I be in my new house place daydream',
 'patch up the crack along the wall',
 'I pass it and lose track of what I be say',
 "'cause that be where I be when I get the call",
 "when the first stone 's throw there be screamin",
 'in the street there be a ragin riot',
 'when it be burn the bitch they be shriek',
 'when the truth come out it be quiet',
 'so they kill Cassandra first',
 "'cause she fear the bad"]

## N-gram vectorizer on song level: identifying identical lines and leaving only one

In [7]:
corpus = dataset.Lyrics_lemmatized.iloc[0]
ngram_vectorizer = CountVectorizer(analyzer='word', ngram_range=(2, 3))
X2 = ngram_vectorizer.fit_transform(corpus)
X2 = X2.toarray()
similarity_matrix_song = cosine_similarity(X2)
similarity_matrix_song.shape

(51, 51)

In [8]:
corpus_hover = []
for x in range(len(corpus)):
    corpus_hover.append(
        list(map(lambda l: f'<br>X={l}</br><br>Y={corpus[x]}</br>',
         corpus))
    )

In [9]:
fig = px.imshow(
    similarity_matrix_song,
    labels=dict(x="X", y="Y", color="Intensity"),
    x=[i for i in range(similarity_matrix_song.shape[0])],
    y=[i for i in range(similarity_matrix_song.shape[0])],
    #text_auto=True,                # Automatically displays values inside the boxes
    color_continuous_scale="Viridis" # Choose color mapping style
)

fig.update_traces(
    customdata=corpus_hover,
    hovertemplate="<br>X=%{x}; Y=%{y}</br>%{customdata}<br>Similarity:%{z}</br><extra></extra>"
)
fig.show()

In [10]:
def delete_reduntant_lines(loc):
    ngram_vectorizer = CountVectorizer(analyzer='word', ngram_range=(2, 3))
    X2 = ngram_vectorizer.fit_transform(loc)
    X2 = X2.toarray()
    similarity_matrix_song = cosine_similarity(X2)

    #lines_stay = []
    lines_delete = np.array([])
    appended_lines = []
    
    for line_i in range(X2.shape[0]):
        similarity_line = similarity_matrix_song[line_i]
        similar_lines = np.nonzero(similarity_line > 0.9)[0]
        #print(line_i, similar_lines)
        if len(similar_lines) > 1:
            if similar_lines[0] not in appended_lines:
                #lines_stay.append(similar_lines[0])
                lines_delete = np.append(lines_delete, similar_lines[1:])
                appended_lines += list(similar_lines)
    array = np.arange(0, loc.__len__())
    lines_to_save = array[np.isin(array, lines_delete,
              invert=True)]
    return [loc[i] for i in lines_to_save]
song_no_redundant = delete_reduntant_lines(dataset.Lyrics_lemmatized.iloc[0])

In [11]:
X_new = ngram_vectorizer.fit_transform(song_no_redundant)
similarity_matrix_song_new = cosine_similarity(X_new)

corpus_hover_new = []
for x in range(len(song_no_redundant)):
    corpus_hover_new.append(
        list(map(lambda l: f'<br>X={l}</br><br>Y={song_no_redundant[x]}</br>',
         song_no_redundant))
    )

fig = px.imshow(
    similarity_matrix_song_new,
    labels=dict(x="X", y="Y", color="Intensity"),
    x=[i for i in range(similarity_matrix_song_new.shape[0])],
    y=[i for i in range(similarity_matrix_song_new.shape[0])],
    #text_auto=True,                # Automatically displays values inside the boxes
    color_continuous_scale="Viridis" # Choose color mapping style
)

fig.update_traces(
    customdata=corpus_hover_new,
    hovertemplate="<br>X=%{x}; Y=%{y}</br>%{customdata}<br>Similarity:%{z}</br><extra></extra>"
)
fig.show()

In [16]:
dataset['Lyrics_no_redundant'] = dataset.Lyrics.apply(lambda x: x.split('\n')).apply(delete_reduntant_lines).apply(lambda x: '\n'.join(x))

In [18]:
dataset.to_csv('.data/dataset_no_reduntant.csv', index=False)